<a href="https://colab.research.google.com/github/leman-cap13/NLP_projects/blob/main/LORA_PEFT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Lora

In [ ]:
!pip uninstall -y torchao torchvision -q
!pip install -q -U transformers peft datasets accelerate evaluate scikit-learn

In [ ]:
import torch
import numpy as np
import pandas as pd

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

from peft import LoraConfig, get_peft_model, TaskType #SEQ_2_SEQ_LM. CAUSAL_LM. SEQ_CLS

from sklearn.metrics import accuracy_score, f1_score, classification_report

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
dataset = load_dataset("dair-ai/emotion")

In [ ]:
dataset

In [ ]:
dataset["train"].column_names

In [ ]:
label_names = dataset["train"].features["label"].names
label_names

In [ ]:
num_labels = len(label_names)
num_labels

In [ ]:
id2label = {i: label for i, label in enumerate(label_names)}
label2id = {label: i for i, label in enumerate(label_names)}

In [ ]:
id2label

In [ ]:
train_dataset = dataset["train"].shuffle(seed=42).select(range(10000))
val_dataset = dataset["validation"].shuffle(seed=42).select(range(1000))
test_dataset = dataset["test"].shuffle(seed=42).select(range(1000))

In [ ]:
train_dataset

In [ ]:
val_dataset

In [ ]:
test_dataset

In [ ]:
MODEL_ID = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

In [ ]:
def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=128
    )

tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

tokenized_val = val_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

tokenized_test = test_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

In [ ]:
tokenized_train = tokenized_train.rename_column("label", "labels")
tokenized_val = tokenized_val.rename_column("label", "labels")
tokenized_test = tokenized_test.rename_column("label", "labels")

tokenized_train.set_format("torch")
tokenized_val.set_format("torch")
tokenized_test.set_format("torch")



In [ ]:
tokenized_train.features

In [ ]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
    )

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

model.to(device)


In [ ]:
def print_trainable_parameters(model):
    total_params = 0
    trainable_params = 0

    for param in model.parameters():
        total_params += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()

    trainable_percent = 100 * trainable_params / total_params

    print("Total parameters:", total_params)
    print("Trainable parameters:", trainable_params)
    print(f"Trainable percent: {trainable_percent:.4f}%")

print("Before LoRA:")
print_trainable_parameters(model)

In [ ]:
import torch.nn as nn

for name, module in model.named_modules():
    if isinstance(module, nn.Linear):
        print(name)

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_lin", "v_lin"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS,
    modules_to_save=["pre_classifier", "classifier"]
)

model = get_peft_model(model, lora_config)

print("After LoRA:")
model.print_trainable_parameters()

In [ ]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, param.shape)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, predictions)
    f1_macro = f1_score(labels, predictions, average="macro")
    f1_weighted = f1_score(labels, predictions, average="weighted")

    return {
        "accuracy": acc,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="distilbert_emotion_lora",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=20,
    report_to="none",
    fp16=torch.cuda.is_available(),
    dataloader_pin_memory=False
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:
print("Evaluation before LoRA training:")
before_metrics = trainer.evaluate()
before_metrics

In [ ]:
train_result = trainer.train()

print(train_result)

In [ ]:
print("Evaluation after LoRA training:")
after_metrics = trainer.evaluate()
after_metrics

In [ ]:
test_metrics = trainer.evaluate(tokenized_test)

print("Test metrics:")
test_metrics

In [ ]:
def predict_emotion(text):
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    pred_id = torch.argmax(logits, dim=-1).item()
    pred_label = id2label[pred_id]

    probs = torch.softmax(logits, dim=-1)[0].detach().cpu().numpy()

    return pred_label, probs


examples = [
    "I am so happy and excited today!",
    "I feel very lonely and sad.",
    "I am terrified of what might happen.",
    "I love this so much.",
    "I am really angry about this situation.",
    "Wow, I did not expect that at all!"
]

for text in examples:
    label, probs = predict_emotion(text)
    print("\nText:", text)
    print("Predicted emotion:", label)

In [ ]:
predictions = trainer.predict(tokenized_test)

logits = predictions.predictions
true_labels = predictions.label_ids
pred_labels = np.argmax(logits, axis=-1)

print(classification_report(
    true_labels,
    pred_labels,
    target_names=label_names
))

In [ ]:
logs = trainer.state.log_history

loss_rows = []

for log in logs:
    if "loss" in log:
        loss_rows.append({
            "step": log["step"],
            "loss": log["loss"],
            "type": "train"
        })

    if "eval_loss" in log:
        loss_rows.append({
            "step": log["step"],
            "loss": log["eval_loss"],
            "type": "eval"
        })

loss_df = pd.DataFrame(loss_rows)
loss_df

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

for loss_type in loss_df["type"].unique():
    sub_df = loss_df[loss_df["type"] == loss_type]
    plt.plot(sub_df["step"], sub_df["loss"], marker="o", label=loss_type)

plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("LoRA Emotion Classification Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
comparison = pd.DataFrame([
    {
        "stage": "Before LoRA training",
        "eval_loss": before_metrics["eval_loss"],
        "accuracy": before_metrics["eval_accuracy"],
        "f1_macro": before_metrics["eval_f1_macro"],
        "f1_weighted": before_metrics["eval_f1_weighted"]
    },
    {
        "stage": "After LoRA training",
        "eval_loss": after_metrics["eval_loss"],
        "accuracy": after_metrics["eval_accuracy"],
        "f1_macro": after_metrics["eval_f1_macro"],
        "f1_weighted": after_metrics["eval_f1_weighted"]
    }
])

comparison

In [ ]:
plt.figure(figsize=(7, 5))

plt.bar(comparison["stage"], comparison["accuracy"])
plt.ylabel("Accuracy")
plt.title("Accuracy Before vs After LoRA Training")
plt.xticks(rotation=20)
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = confusion_matrix(true_labels, pred_labels)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=label_names
)

plt.figure(figsize=(8, 6))
disp.plot(values_format="d")
plt.title("Confusion Matrix - LoRA Emotion Classification")
plt.xticks(rotation=45)
plt.show()

In [ ]:
model.save_pretrained("distilbert_emotion_lora_adapter")
tokenizer.save_pretrained("distilbert_emotion_lora_adapter")


In [ ]:
merged_model = model.merge_and_unload()

merged_model.save_pretrained("final_emotion_merged_model")
tokenizer.save_pretrained("final_emotion_merged_model")

In [ ]:
model_path = "final_emotion_merged_model"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

model.eval()